# Math 638 Project 2 - Homework 5.1

## Syllable-Level Neural Decipherment with Partial Anchors and an Independent English Reference Corpus

This report documents the complete Homework 5.1 experiment from corpus construction through LLM criticism. It is written as a reproducible report for the submitted `codes/`, `data/`, and `result images/` folders.

The central experimental question is:

$$
\text{Can a hidden syllable substitution be recovered from partial anchor translations,}
$$

$$
\text{the structure of the glyph corpus, and independent English statistics?}
$$

The experiment uses Marcus Aurelius' *Meditations* as the hidden source text and Benjamin Jowett's translation of Plato's *Republic* as the independent English reference corpus.

All numerical claims in this report come from generated CSV files, saved trained-model outputs, or the post-hoc passage evidence file `data/task10_decoded_passage_examples.csv`. The hidden map is kept separate from the fitting inputs and is opened only for final evaluation. In the proofchecked training notebook, the held-out test glyph sequence is also not loaded until all fitting choices and model states have been fixed.

## Executive Summary

The final cleaned corpora contain all 12 books of *Meditations* and all 10 books of *The Republic*. After the frozen written-syllable tokenization procedure, *Meditations* contains 77,260 syllable occurrences and 3,052 distinct source syllable types, while *The Republic* contains 157,157 syllable occurrences and 4,410 distinct syllable types.

A seeded random bijection maps the 3,052 source syllables to 3,052 neutral glyph labels. The glyph sequence is split contiguously into 70% training, 15% validation, and 15% test data. Nested anchor sets reveal approximately 5%, 10%, 20%, and 40% of the inverse map.

The Republic-informed system uses a two-stage procedure. First, a fixed smoothed Republic bigram model and the training glyph sequence are used to infer a candidate substitution map under the anchor constraints. Second, an embedding-plus-MLP neural decoder is trained on the inferred labels. This is explicitly reported as a two-stage method; the neural network is not claimed to have independently performed the decipherment.

The main empirical conclusions are conservative:

- The complete Republic-informed two-stage pipeline materially improves decoding over the anchor-only baseline. Because the informed pipeline also supplies inferred pseudo-labels at unanchored positions, this comparison attributes the improvement to the full Republic-informed stage; it does not isolate English statistics from the additional pseudo-label supervision.
- At 20% anchors with the medium decoder and context radius 2, overall test accuracy rises from 14.36% for anchor-only training to 26.40% for Republic-informed training.
- Unanchored-token accuracy is still only 9.37% in that configuration.
- Increasing anchors has the clearest positive effect. With 40% anchors, Republic-informed unanchored-token accuracy reaches 15.72%.
- Context widths 3, 5, and 9 produce nearly identical Republic-informed accuracy, so the tested context range is not the principal bottleneck.
- Increasing decoder size gives only a modest improvement, while low-frequency glyphs remain extremely difficult.
- Classical recovery of the unanchored substitution map remains below 1% for every anchor level tested. Only 44.72% of glyph types appear as valid test centers, so all-$$K$$ neural map recovery conservatively counts unobserved types as unrecovered; observed-type recovery is reported separately in the appendix.
- The LLM critic correctly recognized very weak predictions as speculative, but it substantially overestimated the reliability of high-confidence predictions: only 2 of 12 translations labeled strongly supported were actually correct.

The experiment therefore demonstrates partial benefit from independent English statistics, but not reliable recovery of the hidden unanchored map.

# 1. Editions and Cleaning Procedures

## 1.1 Source editions

| Corpus | Author | Translator | Project Gutenberg eBook | Books retained | Characters | Words | Paragraphs |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Meditations | Marcus Aurelius | Meric Casaubon | 2680 | 12 | 312,633 | 56,941 | 417 |
| The Republic | Plato | Benjamin Jowett | 1497 | 10 | 641,395 | 118,911 | 4,308 |

The exact public-domain sources were:

- Marcus Aurelius, *Meditations*, translated by Meric Casaubon, Project Gutenberg eBook #2680: [Project Gutenberg - Meditations](https://www.gutenberg.org/cache/epub/2680/pg2680-images.html)
- Plato, *The Republic*, translated by Benjamin Jowett, Project Gutenberg eBook #1497: [Project Gutenberg - The Republic](https://www.gutenberg.org/cache/epub/1497/pg1497-images.html)

The complete primary texts were used, not selected passages.

## 1.2 Cleaning procedure

The same general cleaning policy was applied to both corpora:

- remove Project Gutenberg boilerplate and licensing material from the linguistic corpus;
- remove editorial front matter and back matter that are not part of the primary text;
- preserve ordinary punctuation and word boundaries;
- normalize curly quotation marks and apostrophes to plain ASCII forms;
- normalize long dashes to a standard hyphen;
- collapse repeated whitespace;
- convert the working English text to lowercase;
- retain every required book;
- use the actual book headings to verify extraction boundaries;
- exclude the structural book headings from the linguistic token stream.

For *Meditations*, the extraction retained `THE FIRST BOOK` through `THE TWELFTH BOOK` and stopped before the appendix. For *The Republic*, it retained `BOOK I.` through `BOOK X.` and excluded Jowett's introduction and other front matter.

The extraction was verified book by book before the combined corpora were saved. SHA-256 hashes are recorded in `data/corpus_metadata.csv`, so later runs can confirm that exactly the same cleaned corpora are being used.

# 2. Syllable-Tokenization Program, Rules, Resources, Examples, and Reconstruction

## 2.1 Written-syllable model

Each normalized English word is represented as a concatenation of written syllable segments:

$$
w = s_1 s_2 \cdots s_m.
$$

The implementation requires exact reconstruction:

$$
s_1 + s_2 + \cdots + s_m = w.
$$

This condition verifies that the tokenizer did not lose, duplicate, or reorder characters. It does not imply that the written boundaries are phonetically unique or linguistically perfect.

## 2.2 Tokenization resource

The tokenizer uses `Pyphen` version 0.18.1 with the `en_US` hyphenation patterns. This is treated as a reproducible written-syllable approximation.

The same tokenization rule is applied to both corpora.

Before syllabification, historical Latin ligatures are normalized consistently. In particular:

- `æ` becomes `ae`;
- `œ` becomes `oe`;
- common typographic ligatures such as `ﬁ` and `ﬂ` are expanded.

This correction was necessary because the Casaubon text contains historical spellings such as `Cæsar`, `Mæcenas`, and `Crœsus`.

A deterministic vowel-based fallback exists for cases in which the primary method fails to reconstruct a word. In the final run, the fallback was used 0 times in *Meditations* and 0 times in *The Republic*.

Remaining Greek alphabetic fragments in *Meditations* are not treated as English syllable tokens by the ASCII English-word tokenizer. They are documented as nonsyllabic artifacts rather than silently merged into an unknown token.

In the final tokenization text this amounts to **31 Greek-word occurrences, 26 distinct Greek word forms, and 231 non-ASCII alphabetic characters**. They remain visible non-token spans and are excluded from the English syllable counts. This is a documented tokenization limitation; a different cleaning convention could instead remove or transliterate them, but that alternative was not tested.

## 2.3 Representative examples

| Word | Written-syllable segmentation | Reconstructed word | Fallback used |
| --- | --- | --- | --- |
| philosophy | phi \| los \| o \| phy | philosophy | False |
| justice | jus \| tice | justice | False |
| nature | na \| ture | nature | False |
| reason | rea \| son | reason | False |
| virtue | virtue | virtue | False |
| socrates | socrates | socrates | False |
| knowledge | knowl \| edge | knowledge | False |
| unconscionableness | un \| con \| scionable \| ness | unconscionableness | False |
| incomprehensible | in \| com \| pre \| hen \| si \| ble | incomprehensible | False |
| prestidigitators | pres \| tidig \| i \| ta \| tors | prestidigitators | False |

The reconstruction check reported:

- *Meditations*: 0 reconstruction failures;
- *The Republic*: 0 reconstruction failures.

Thus every distinct normalized English word processed by the tokenizer reconstructed exactly.

# 3. Syllable Counts, Vocabulary Sizes, Frequency Tables, IDs, and Missing Reference Syllables

## 3.1 Corpus-level counts

| Corpus | Total syllable occurrences | Distinct syllable types |
| --- | --- | --- |
| Meditations | 77,260 | 3,052 |
| The Republic | 157,157 | 4,410 |

For the source corpus,

$$
L = 77{,}260
$$

and

$$
K = |T| = 3{,}052.
$$

The two corpora contain:

- 2,240 shared syllable types;
- 812 source-only syllable types;
- 2,170 reference-only syllable types;
- 5,222 syllable types in the union vocabulary.

## 3.2 Source IDs

Every distinct source syllable receives one permanent integer ID in `data/source_syllable_vocabulary.csv`. The ID assignment is never changed in later stages.

A sample of the most frequent source syllables is:

| Syllable | Meditations frequency | Republic frequency |
| --- | --- | --- |
| and | 2,863 | 5,755 |
| to | 2,243 | 3,599 |
| of | 1,979 | 4,750 |
| the | 1,861 | 7,103 |
| that | 1,820 | 1,963 |
| in | 1,333 | 3,402 |
| er | 1,332 | 1,838 |
| is | 1,272 | 2,641 |
| it | 1,213 | 312 |
| be | 1,211 | 2,671 |
| a | 905 | 2,134 |
| so | 866 | 611 |
| as | 838 | 1,163 |
| thou | 811 | 13 |
| ly | 806 | 1,700 |

The complete source frequency table and source syllable-to-ID table are submitted in:

- `data/source_syllable_vocabulary.csv`

The complete union table is submitted in:

- `data/union_syllable_vocabulary.csv`

## 3.3 Source syllables absent from the reference corpus

No source syllable is replaced by a common unknown symbol.

If distinct source syllables $$a$$ and $$b$$ do not occur in *The Republic*, then

$$
\operatorname{ID}(a) \ne \operatorname{ID}(b),
$$

while their empirical Republic frequencies are simply

$$
f_R(a)=0,
\qquad
f_R(b)=0.
$$

Therefore missing reference coverage never merges distinct source symbols. In the later English reference model, zero empirical counts are handled by additive smoothing rather than by an `<UNK>` collapse.

# 4. Hidden Glyph Map

A neutral artificial glyph vocabulary

$$
G = \{g_1,\ldots,g_K\}
$$

was created with

$$
|G|=|T|=3{,}052.
$$

Using random seed `63803`, a random permutation defines the bijection

$$
\Phi:T\rightarrow G.
$$

The same map is used for every occurrence of every source syllable.

Glyph names have the neutral form `G0000`, `G0001`, and so on. Their names contain no syllable information.

A small sample of the hidden map is shown only for report documentation:

| Source ID | Syllable | Glyph ID | Glyph label |
| --- | --- | --- | --- |
| 0 | a | 2828 | G2828 |
| 1 | ab | 1971 | G1971 |
| 2 | aban | 128 | G0128 |
| 3 | aber | 1454 | G1454 |
| 4 | abid | 2793 | G2793 |
| 5 | abide | 468 | G0468 |
| 6 | abides | 2690 | G2690 |
| 7 | abideth | 527 | G0527 |
| 8 | abil | 105 | G0105 |
| 9 | able | 1372 | G1372 |

The complete map is stored separately at:

`data/evaluation_only/hidden_syllable_glyph_bijection.csv`

It is not stored with the fitting inputs.

The fitting-side files are instead placed in `data/decipherment_inputs/` and contain glyph identities without the hidden plaintext translations.

Spaces, punctuation, apostrophes, hyphens, and newlines are copied unchanged by the cipher. They remain visible separators but are not syllable tokens and are excluded from token counts, anchor percentages, and decoding accuracy.

# 5. Revealed Anchor Maps and Actual Percentages

A second fixed random seed, `63845`, was used to generate one random ordering of the distinct glyph types. Each anchor set is a prefix of that same ordering, so the anchor sets are nested:

$$
A_5 \subseteq A_{10} \subseteq A_{20} \subseteq A_{40}.
$$

The actual anchor counts and percentages are:

| Requested percentage | Revealed glyph types | Actual percentage |
| --- | --- | --- |
| 5% | 153 | 5.0131% |
| 10% | 305 | 9.9934% |
| 20% | 610 | 19.9869% |
| 40% | 1,221 | 40.0066% |

The revealed maps are submitted as:

- `data/decipherment_inputs/anchors/anchors_05.csv`
- `data/decipherment_inputs/anchors/anchors_10.csv`
- `data/decipherment_inputs/anchors/anchors_20.csv`
- `data/decipherment_inputs/anchors/anchors_40.csv`

These files are the only direct glyph-to-syllable translations supplied to the learning procedure.

# 6. Training, Validation, and Test Partitions

The 77,260-token glyph sequence is divided contiguously into 70% training, 15% validation, and 15% test data.

| Partition | First global token | Last global token | Token count | Fraction |
| --- | --- | --- | --- | --- |
| train | 0 | 54081 | 54,082 | 70.0% |
| validation | 54082 | 65670 | 11,589 | 15.0% |
| test | 65671 | 77259 | 11,589 | 15.0% |

The partition files are:

- `data/decipherment_inputs/splits/train_glyph_sequence.csv`
- `data/decipherment_inputs/splits/validation_glyph_sequence.csv`
- `data/decipherment_inputs/splits/test_glyph_sequence.csv`

No shuffling of token positions is used.

For a context radius $$m$$, a valid center position must satisfy

$$
m \le i < N-m
$$

inside its own partition. Windows are generated separately within each partition, so no context window can cross a train/validation/test boundary.

The verified valid-center counts are:

| Partition | Context radius m | Window width | Valid centers |
| --- | --- | --- | --- |
| train | 1 | 3 | 54,080 |
| validation | 1 | 3 | 11,587 |
| test | 1 | 3 | 11,587 |
| train | 2 | 5 | 54,078 |
| validation | 2 | 5 | 11,585 |
| test | 2 | 5 | 11,585 |
| train | 4 | 9 | 54,074 |
| validation | 4 | 9 | 11,581 |
| test | 4 | 9 | 11,581 |

# 7. English Reference Model, Objective, and Contribution to Decipherment

## 7.1 Fixed Republic bigram model

Only *The Republic* is used to estimate independent English sequence statistics.

For source syllables $$a,b\in T$$, the fixed reference model is

$$
P_R(b\mid a)
=
\frac{C_R(a,b)+\alpha}
{C_R(a)+\alpha K},
$$

with additive smoothing

$$
\alpha = 0.10.
$$

The underlying count model is the conditional maximum-likelihood bigram model. Before smoothing, its negative log-likelihood on the usable Republic bigrams is

$$
L_R(P)=-\sum_{a,b\in T} C_R(a,b)\log P(b\mid a),
$$

subject to $$\sum_b P(b\mid a)=1$$ for each $$a$$. Additive smoothing is then applied to the empirical conditional counts. Of the 157,157 Republic syllable occurrences, 149,831 (95.34%) belong to the source vocabulary $$T$$, and 143,006 adjacent bigrams have both endpoints in $$T$$ and enter the reference bigram counts.

A Republic syllable outside the source vocabulary is not merged into a source token. It is simply outside the decoder output vocabulary. A source syllable absent from *The Republic* has zero empirical count but receives nonzero smoothed probability.

## 7.2 Classical candidate-map objective

For a candidate substitution map

$$
\pi:G\rightarrow T,
$$

the training glyph sequence provides glyph bigram counts $$C_G(g,h)$$. The candidate map is scored by

$$
S(\pi)
=
\sum_{g,h}
C_G(g,h)
\log P_R(\pi(h)\mid\pi(g)).
$$

The anchor translations are fixed constraints.

The remaining map is initialized by matching unanchored glyph-frequency ranks with Republic syllable-frequency ranks. A seeded swap search then optimizes the 200 most frequent unanchored glyphs for 6,000 swap proposals. Thus this is not a global search over every unanchored type. At 20% anchors, 2,442 glyph types are unanchored, so the swap refinement directly searches only 200/2,442 = 8.19% of them; the remaining unanchored assignments stay at their initial frequency-rank pairing because the swap proposals are restricted to the top-200 set.

The resulting score changes were:

| Anchor level | Initial bigram score | Final bigram score | Improvement |
| --- | --- | --- | --- |
| 5% | -419,269.2 | -388,192.9 | 31,076.3 |
| 10% | -417,339.6 | -387,418.0 | 29,921.6 |
| 20% | -408,839.9 | -379,703.9 | 29,136.0 |
| 40% | -399,133.4 | -368,584.7 | 30,548.8 |

## 7.3 What the reference information contributes

Without an English reference, the names of unanchored glyphs are permutation-symmetric: the observed glyph sequence alone does not identify which English syllable name should be attached to an unrevealed glyph.

The Republic model supplies an external preference for candidate English labels based on independent English frequency and bigram structure.

However, the final results show that this information is insufficient for reliable recovery of the unseen substitution map. Classical unanchored-map recovery remains below 1% at every tested anchor percentage. The Republic therefore contributes useful statistical pressure, but it does not resolve most unanchored identities.

# 8. Neural Decoder Architecture and Trainable Parameters

The neural decoder uses an embedding layer followed by a multilayer perceptron. No Transformer or attention mechanism is used.

For context radius $$m$$, the input is

$$
y_{i-m},\ldots,y_i,\ldots,y_{i+m}.
$$

Each glyph ID is embedded into $$d$$ dimensions. The $$2m+1$$ embeddings are concatenated and passed through a hidden ReLU layer of width $$h$$. The output layer has $$K=3{,}052$$ logits, one for each possible source syllable.

The number of trainable parameters is

$$
Kd + ((2m+1)d)h + h + hK + K.
$$

The tested architectures were:

| Configuration | Window width | Embedding dimension | Hidden dimension | Trainable parameters |
| --- | --- | --- | --- | --- |
| Small, m=2 | 5 | 16 | 64 | 252,396 |
| Medium, m=1 | 3 | 32 | 128 | 503,788 |
| Medium, m=2 | 5 | 32 | 128 | 511,980 |
| Medium, m=4 | 9 | 32 | 128 | 528,364 |
| Large, m=2 | 5 | 64 | 256 | 1,061,868 |

The parameter counts were recorded from the actual PyTorch models and saved in `data/task6_8_training_manifest.csv`.

# 9. Decoder Loss Function and Training Procedure

The experiment deliberately separates classical inference from subsequent neural training.

## 9.1 Anchor-only neural baseline

Let $$I_A$$ be the set of training positions whose center glyph has a revealed anchor translation $$a(y_i)$$.

The anchor-only loss is

$$
L_{\text{anchor}}
=
-\frac{1}{|I_A|}
\sum_{i\in I_A}
\log
P_\theta
\left(
a(y_i)
\mid
y_{i-m},\ldots,y_{i+m}
\right).
$$

Only revealed anchor translations supply labels. Consequently, unanchored output classes receive no direct supervised labels in this baseline.

## 9.2 Republic-informed two-stage model

First, Stage A infers a fixed candidate substitution map

$$
\widehat{\pi}:G\rightarrow T
$$

using the training glyph bigrams, the fixed Republic bigram model, and the revealed anchor constraints.

Second, the neural decoder is trained on all valid training positions using the inferred map as pseudo-labels:

$$
L_{\text{informed}}
=
-\frac{1}{|I|}
\sum_{i\in I}
\log
P_\theta
\left(
\widehat{\pi}(y_i)
\mid
y_{i-m},\ldots,y_{i+m}
\right).
$$

There is no additional hidden-map loss term. The true map is not used in either training objective.

This distinction is important: the Republic-informed neural decoder is learning the labels produced by the classical inference stage. Therefore the experiment does not claim that the neural network itself performed the original decipherment.

## 9.3 Optimization and model selection

The neural training settings were fixed before test evaluation:

- optimizer: Adam;
- learning rate: `1e-3`;
- weight decay: `1e-5`;
- batch size: `1024`;
- maximum epochs: `8`;
- validation early-stopping patience: `2`;
- model selection criterion: validation cross-entropy.

The test partition and the hidden true map are not used for fitting or model selection. All models are fixed before final test evaluation begins.

**Proofcheck transparency note.** In the original development version of the Steps 6-10 notebook, `test_glyph_sequence.csv` was loaded into a Python variable before training, although static dependency checking confirms that the test sequence was not referenced by the classical inference, training loss, validation loss, early stopping, or hyperparameter definitions. This still violated the assignment's stricter requirement that the test partition remain unavailable. The proofchecked notebook therefore defers loading the test sequence until the final-evaluation section. The existing numerical results are retained because no test value entered fitting or selection.

## 9.4 Information-flow and validation-loss interpretation

The proofchecked implementation loads only the training and validation glyph sequences before fitting. The held-out test glyph sequence and the complete hidden inverse map are loaded only after every model state and hyperparameter sweep has been fixed.

For the Republic-informed model, validation cross-entropy is computed against the **inferred pseudo-label map**, not against the hidden true translations. Therefore a low Republic-informed validation loss demonstrates that the MLP learns and generalizes its permitted pseudo-label target; it is not, by itself, evidence that the decipherment is correct. Hidden-map accuracy is evaluated only in the final evaluation stage.

# 10. Anchor-Only Comparison, Accuracy Sweeps, and Failure Modes

## 10.0 Test metrics and extraction of a single inverse map

Token accuracies are computed over valid held-out test centers only. To convert contextual predictions into a single candidate inverse map, all predictions for the same center glyph are collected and the **majority-vote source token** is assigned to that glyph. Ties are broken deterministically by the smaller source-token ID. A glyph type that never appears as a valid test center receives no candidate and is counted as unrecovered in the all-$$K$$ map metric.

The test partition covers 44.72% of the 3,052 glyph types as valid centers. Consequently, the all-$$K$$ map-recovery metric is deliberately conservative; the appendix also reports recovery restricted to observed test glyph types.

## 10.1 Anchor-percentage sweep

The anchor sweep uses the medium decoder with context radius $$m=2$$.

| Anchors | Anchor-only overall | Republic-informed overall | Republic-informed unanchored | Republic-informed decoder map recovery (all K) | Classical unanchored map recovery |
| --- | --- | --- | --- | --- | --- |
| 5% | 3.07% | 8.83% | 3.61% | 2.16% | 0.28% |
| 10% | 4.59% | 16.25% | 7.76% | 4.13% | 0.66% |
| 20% | 14.36% | 26.40% | 9.37% | 6.85% | 0.37% |
| 40% | 31.10% | 44.66% | 15.72% | 13.70% | 0.93% |

![Token accuracy versus anchor percentage](../result%20images/token_accuracy_vs_anchor_percentage.png)

![Unanchored accuracy versus anchor percentage](../result%20images/unanchored_accuracy_vs_anchor_percentage.png)

![Decoder map recovery versus anchor percentage](../result%20images/map_recovery_vs_anchor_percentage.png)

![Classical map recovery versus anchor percentage](../result%20images/classical_map_recovery_vs_anchor_percentage.png)

The strongest observed factor is anchor percentage. Republic-informed unanchored-token accuracy rises from 3.61% at 5% anchors to 15.72% at 40% anchors.

The anchor-only baseline has 0% unanchored-token accuracy at every anchor level. This is expected from its training objective: it receives labels only for anchored classes.

The static classical map itself performs poorly on unseen types. Even at 40% anchors, classical recovery restricted to unanchored glyph types is only 0.93%. Most of the all-type classical map accuracy therefore comes directly from the revealed anchors.

## 10.2 Context-window sweep

The context sweep uses 20% anchors and the medium decoder.

| Context radius m | Window width | Anchor-only overall | Republic-informed overall | Republic-informed unanchored | Republic-informed map recovery |
| --- | --- | --- | --- | --- | --- |
| 1 | 3 | 14.72% | 26.42% | 9.37% | 6.75% |
| 2 | 5 | 14.36% | 26.40% | 9.37% | 6.85% |
| 4 | 9 | 14.35% | 26.34% | 9.37% | 6.62% |

![Token accuracy versus context window](../result%20images/token_accuracy_vs_context_window.png)

Across widths 3, 5, and 9, Republic-informed overall accuracy stays near 26.3%-26.4%, and unanchored accuracy stays near 9.37%. Therefore the tested context sizes do not materially change the result.

## 10.3 Decoder-size sweep

The decoder-size sweep uses 20% anchors and context radius $$m=2$$.

| Decoder size | Parameters | Anchor-only overall | Republic-informed overall | Republic-informed unanchored | Republic-informed map recovery |
| --- | --- | --- | --- | --- | --- |
| Small | 252,396 | 8.96% | 25.34% | 9.37% | 4.72% |
| Medium | 511,980 | 14.36% | 26.40% | 9.37% | 6.85% |
| Large | 1,061,868 | 17.74% | 26.66% | 9.37% | 7.44% |

![Token accuracy versus decoder size](../result%20images/token_accuracy_vs_decoder_parameters.png)

Increasing model size raises Republic-informed overall accuracy only modestly, from 25.34% to 26.66%. Unanchored-token accuracy remains essentially unchanged at 9.37%. Map recovery improves somewhat, from 4.72% to 7.44%.

These results do not support decoder capacity as the main bottleneck over the tested range.

## 10.4 Stage-A versus Stage-B diagnostic

For the central 20%-anchor, $$m=2$$, medium configuration, the static classical inferred map itself has 9.37% unanchored token accuracy on the test occurrences. The trained neural decoder also has 9.37% unanchored truth accuracy and agrees with the static pseudo-label map on 95.64% of unanchored test positions.

This confirms the interpretation required by the assignment: Stage B is primarily learning the Stage-A candidate naming rather than discovering a substantially different unanchored decipherment. The nearly unchanged unanchored accuracy across context widths and decoder sizes is consistent with this two-stage structure. The detailed numbers are saved in `data/task9_stage_agreement_diagnostics.csv`.

## 10.5 Observed failure modes

The fixed reference configuration for failure analysis is:

- 20% anchors;
- context radius $$m=2$$;
- medium decoder;
- Republic-informed two-stage method.

| Category | Group | Evaluated positions | Accuracy |
| --- | --- | --- | --- |
| anchor_status | anchored | 2,277 | 96.00% |
| anchor_status | unanchored | 9,308 | 9.37% |
| reference_coverage | true syllable appears in Republic | 11,256 | 26.57% |
| reference_coverage | true syllable absent from Republic | 329 | 20.36% |
| training_glyph_frequency | lowest frequency quartile | 416 | 0.48% |
| training_glyph_frequency | second frequency quartile | 126 | 12.70% |
| training_glyph_frequency | third frequency quartile | 588 | 18.20% |
| training_glyph_frequency | highest frequency quartile | 10,455 | 28.05% |

Several failures are directly observed:

- **Anchor scarcity matters.** Accuracy rises substantially as the revealed percentage increases.
- **Unanchored symbols remain difficult.** In the fixed configuration, anchored occurrences are decoded at 96.00%, while unanchored occurrences are decoded at only 9.37%.
- **Rare glyphs are extremely difficult.** The lowest training-frequency quartile has only 0.48% accuracy, compared with 28.05% in the highest quartile.
- **Reference coverage is associated with higher accuracy, but the all-position comparison is observational.** Positions whose true syllable appears in *The Republic* reach 26.57% accuracy, versus 20.36% when the true syllable is absent. Because anchor composition and token frequency can differ between these groups, this comparison should not be read as a clean causal estimate of the benefit of Republic coverage.
- **The tested context range is not a major limitation.** Widths 3, 5, and 9 perform almost identically.
- **The tested decoder-size range is not the main limitation.** A much larger decoder does not materially improve unanchored accuracy.

Other explanations remain hypotheses rather than established findings:

- too little glyph training text was not tested by subsampling the corpus;
- a smaller token vocabulary was not tested;
- an alternative syllabification scheme was not tested;
- a different or larger independent English reference corpus was not tested.

Those factors should not be presented as demonstrated causes of failure.

# 11. Successful and Failed Decoded Passages

The following examples come from the held-out test partition under the fixed 20%-anchor, $$m=2$$, medium Republic-informed model. The selection is fully reproducible and is saved in `data/task10_decoded_passage_examples.csv`.

For every contiguous 20-token window of valid test centers, the number of correct token predictions is computed. The earliest-starting window is chosen when several windows tie for the maximum or minimum. The selected window is then expanded to complete normalized word boundaries, and the displayed accuracy is recomputed after expansion. This is post-hoc illustration only and is not used for model selection.

## 11.1 Relatively successful passage

Four raw 20-token windows tie for the maximum score of

$$
\frac{12}{20}=60.0\%.
$$

Using the stated earliest-window tie break, no extra token is needed when expanding to complete word boundaries, so the displayed passage remains 12/20 correct.

**Decoded prediction**

> to in nation of in universe?
>
> men contemn per animi, and yet men seek to

**True plaintext**

> to the nature of the universe?
>
> they contemn one another, and yet they seek to

The decoded text preserves several high-frequency fragments and some local structure, but it still misnames important unanchored syllables.

## 11.2 Failed passage

Twenty-five raw 20-token windows tie for the minimum score of 0/20. The earliest such window is selected. Expanding it to complete word boundaries adds one neighboring token, giving displayed accuracy

$$
\frac{1}{21}=4.8\%.
$$

**Decoded prediction**

> eithan per fect, not in imi fect, ing said ac powmoreca tions yesiwould, so the

**True plaintext**

> either one way, or the other way, is not any external thing properly, but that

This example illustrates the central failure mode: the output can contain plausible English-looking fragments while assigning the wrong English names to most unanchored glyphs.

# 12. LLM Prompts and Substantive Responses

The LLM was used as a coding collaborator and, in Step 11, as an experimental critic. The major prompts and response summaries are recorded below; the generated implementation notebooks are the substantive coding artifacts. For Step 11, the exact critic claim list is recorded explicitly because that judgment is itself part of the experiment.

**Record-retention limitation.** For the first four coding interactions, the submitted project preserves the prompts and substantive summaries of the responses rather than complete verbatim chat transcripts. The generated notebooks are the concrete artifacts of those responses. The Step 11 critic prompt and its substantive claim set are preserved explicitly because the timing and content of that response are part of the graded experiment. If the instructor requires verbatim transcripts for Deliverable 12, the original chat exports should be attached; they cannot be reconstructed faithfully from the code alone.

## 12.1 Corpus-preparation prompt

**Prompt**

> Make the ipynb to cite both source cleanly or in markdown styles. Remember this is not LaTeX so it can only do basic markdowns. Don't use special things but stick to $$ for math.

**Substantive response**

The LLM generated the corpus-preparation notebook, documented both Gutenberg sources, separated code and data outputs, and implemented book-by-book extraction and cleaning. During verification, the extraction logic was revised to use explicit book headings rather than a brittle `Transcriber's Notes` stopping rule.

## 12.2 Syllable-modeling prompt

**Prompt**

> First explain the modeling then do the code.

**Substantive response**

The LLM proposed the written-syllable representation, selected Pyphen as a reproducible written-boundary approximation, preserved unique source syllables rather than collapsing them into `<UNK>`, implemented exact reconstruction checks, and produced the Step 2 notebook.

## 12.3 Hidden-map prompt

**Prompt**

> Make sure you follow the instruction closely.

**Substantive response**

The LLM separated fitting inputs from evaluation-only ground truth, generated a seeded hidden bijection, preserved separators, and verified that the glyph sequence exactly matched the Step 2 syllable sequence. A Unicode inconsistency involving `æ` was later detected and corrected by freezing a shared normalization rule across Steps 2 and 3.

## 12.4 Training-pipeline prompt

**Prompt**

> Do steps that should be just one ipynb into like step x-y. Then do the training and verification suggested.

**Substantive response**

The LLM grouped Steps 4-5 and Steps 6-10 by dependency, created the 70/15/15 split and nested anchors, implemented the fixed Republic reference model, built the embedding-plus-MLP decoder, trained anchor-only and Republic-informed models, ran the requested sweeps, evaluated the fixed models, and generated the result figures.

## 12.5 Experimental LLM-critic prompt

After the predictions were fixed and before the hidden unanchored translations were revealed, the following prompt was used:

> You are the critic for a syllable-decipherment experiment. The candidate translations have already been fixed. You may use only the evidence table I provide: glyph ID, candidate syllable, number of test occurrences, majority-vote share, vote margin, mean decoder confidence, training glyph frequency, frequency of the candidate syllable in the independent Republic reference corpus, and whether the candidate agrees with the separately inferred Republic-based classical map. Some glyphs are revealed anchors, but for this critique focus on unanchored recovered translations. Do not assume that model confidence proves correctness. Identify a small group of unanchored translations that are strongly supported by the permitted evidence and a small group that are speculative. Give a short reason for each classification. You are not allowed to see or infer from the hidden answer key.

**Substantive response**

The critic classified 12 unanchored translations as strongly supported because they had repeated occurrences, nearly unanimous votes, high neural confidence, and agreement with the Republic-based classical map. It classified 12 low-support translations as speculative because they had only two or three test occurrences, weak votes, low confidence, and no classical-map agreement.

The critic explicitly warned that the neural decoder and the classical stage could share the same systematic error because both depend on related English evidence. The claims were saved before the hidden answer key was opened. In the proofchecked Step 11 notebook, the exact glyph IDs named in this recorded response are copied explicitly into the machine-readable pre-truth claim file; no secondary hand-weighted support score is allowed to decide which claims count as the LLM's claims.

# 13. Critique of the LLM

## 13.1 What the LLM did well

The LLM was useful in several concrete ways:

- it translated the assignment into a reproducible multi-stage computational pipeline;
- it kept the complete hidden map separate from fitting inputs;
- it enforced nested anchor sets and explicit train/validation/test boundaries;
- it distinguished the classical map-inference stage from the subsequent neural training stage;
- it created reconstruction, bijection, separator, and leakage checks;
- it generated the requested experiments and plots;
- as a critic, it correctly recognized that very sparse, low-confidence predictions were speculative;
- it explicitly warned that agreement between the neural decoder and the Republic-based classical method was not fully independent evidence.

## 13.2 Where the LLM required correction

The LLM also made implementation mistakes that required verification and correction.

### Corpus extraction

An early notebook made the output appear to contain only the first book because headings had been removed from the cleaned corpus and only the beginning of the file was previewed. A later version also used a brittle exact search for `Transcriber's Notes`, which failed on the actual Gutenberg HTML structure.

The correction was to identify every actual `BOOK I.`, `BOOK II.`, and related heading explicitly and verify every book separately.

### Unicode normalization

Step 2 originally used an ASCII `[a-z]+` word rule while Step 3 used Python's broader `.isalpha()` rule. The source contains historical ligatures such as `æ`, so the mismatch produced a `KeyError` for `æ`.

The correction was to freeze a shared preprocessing normalization, including `æ -> ae` and `œ -> oe`, and rerun the tokenization before generating glyphs.

These errors show why LLM-generated code could not be accepted without direct execution and invariance checks. The same principle applies to the critic stage: the proofchecked claim-freezing code preserves the explicit recorded LLM selections rather than replacing the LLM judgment with an undocumented quantitative ranking rule.

## 13.3 Where the Step 11 critic exceeded the evidence

The final truth check is:

| Critic category | Claims | Correct claims | Correct fraction |
| --- | --- | --- | --- |
| strongly supported | 12 | 2 | 16.7% |
| speculative | 12 | 0 | 0.0% |

![LLM critic calibration](../result%20images/task11_llm_critic_calibration.png)

Among the 12 translations labeled strongly supported, only 2 were correct:

- glyph 2748 -> `and`;
- glyph 2196 -> `of`.

Thus

$$
\frac{2}{12}
=
16.7\%.
$$

None of the 12 speculative translations was correct.

The critic was therefore directionally useful at identifying especially weak cases, but the phrase **strongly supported** was too strong. High vote concentration, repeated occurrence, high softmax confidence, and agreement with the classical map mostly measured internal consistency. They did not provide independent identification of the true English label.

This is particularly important because the Republic-informed neural decoder was trained on labels produced by the Republic-based classical stage. Agreement between the two stages is therefore statistically dependent. A wrong permutation can be learned and reproduced with very high confidence.

The correct conclusion is narrower:

> The permitted evidence can rank some predictions as more internally supported than others, but it does not make the high-confidence unanchored translations reliable.

This is the main place where the LLM's claims exceeded what the evidence justified.

# Overall Conclusions

The proofchecked submission implements the assignment's information-restricted decipherment setting, including deferring both the held-out test glyph sequence and the complete hidden inverse map until all fitting choices have been fixed. The final results are intentionally interpreted conservatively.

The artificial syllable cipher is perfectly reproducible, the hidden map is isolated from training, and the evaluation is performed on held-out data only after all models are fixed.

The Republic-informed two-stage pipeline clearly outperforms the anchor-only baseline in the tested configurations. This comparison should be attributed to the full Republic-informed pipeline, which uses English statistics to construct pseudo-labels for unanchored glyphs; it does not isolate the effect of English statistics from the additional pseudo-label supervision. However, the resulting unanchored decoding remains weak. At the central 20%-anchor configuration, only about 9.37% of unanchored test occurrences are decoded correctly. Even at 40% anchors, the unanchored accuracy is only 15.72%.

The strongest manipulated effect is anchor percentage. Among the non-anchor failure diagnostics, low training-glyph frequency shows the clearest performance gradient. Neither a wider context window nor a larger MLP solves the identification problem over the tested ranges. At the central configuration, the neural decoder agrees with the static inferred map on 95.64% of unanchored test positions and has the same 9.37% unanchored truth accuracy as that static map, showing that the classical inference stage dominates the unanchored identification result.

Most importantly, high confidence is not equivalent to decipherment. Both the neural decoder and the LLM critic can become highly confident in a systematically wrong naming of unanchored symbols. Recovery must therefore be judged against the held-out hidden map, not by fluency or plausibility alone.

# Reproducibility and Submitted Files

The project is organized using the required folders:

## `codes/`

- `HW5_1_Step1_Corpus_Preparation.ipynb`
- `HW5_1_Step2_Syllable_Tokenization.ipynb`
- `HW5_1_Step3_Hidden_Glyph_Bijection.ipynb`
- `HW5_1_Steps4-5_Splits_and_Anchors.ipynb`
- `HW5_1_Steps6-10_Training_Evaluation.ipynb`
- `HW5_1_Step11_LLM_Critic.ipynb`
- `HW5_1_Final_Formal_Report_PROOFCHECKED.ipynb`
- `HW5_1_Final_Report_Appendix_PROOFCHECKED.ipynb`

## `data/`

Contains the cleaned corpora, token sequences, vocabulary tables, split definitions, anchor maps, inferred maps, model files, evaluation-only hidden map, and all generated CSV summaries.

## `result images/`

Contains the six requested model-comparison plots plus the LLM-critic calibration figure.

The report notebook uses only basic Markdown headings, tables, links, image references, and `$$...$$` equations so that it remains straightforward to render in VS Code/Jupyter without specialized LaTeX document tooling.